# Module 10 — End-to-End Scikit-learn Pipeline

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scikit-learn, XGBoost, Pandas, NumPy, Plotly, Seaborn, Joblib  
> **Prerequisite**: Modules 1–9 (entire Track 1)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Revolut Churn Dataset](#3-synthetic-revolut-churn-dataset)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Building the Pipeline](#5-building-the-pipeline)
   - 5a. Preprocessing (Imputer, Scaler, Encoder)
   - 5b. Feature Selection
   - 5c. Model (XGBoost)
6. [Cross-Validation](#6-cross-validation)
7. [Hyperparameter Tuning with Pipeline](#7-hyperparameter-tuning-with-pipeline)
8. [Final Model Evaluation](#8-final-model-evaluation)
9. [Model Persistence & Inference](#9-model-persistence--inference)
10. [Visualization Dashboard](#10-visualization-dashboard)
11. [Key Business Takeaway](#11-key-business-takeaway)

---
## §1 · Business Context

### Why do production pipelines matter at Revolut?

A data-science model is **useless** if it can't be deployed. At Revolut:

| Stage | Challenge | Solution |
|-------|-----------|----------|
| **Training** | Data preprocessing, feature engineering, model fitting | Scikit-learn `Pipeline` |
| **Validation** | Prevent data leakage, cross-validation | Pipeline ensures preprocessing is fit on train only |
| **Deployment** | Save model + preprocessing together | `joblib.dump(pipeline)` |
| **Inference** | Apply same transformations to new data | `pipeline.predict(new_data)` |

**The problem without pipelines**:
- Data leakage: test data influences preprocessing (e.g., scaling with test mean).
- Code duplication: separate scripts for training vs. inference.
- Version mismatch: preprocessing changes but model doesn't get retrained.

**The solution**: wrap everything in a single `Pipeline` object that:
1. Preprocesses data (impute, scale, encode)
2. Selects features
3. Trains a model
4. Can be saved, loaded, and used for inference

In this notebook we will:
1. Generate a realistic Revolut churn dataset (mixed types: numeric, categorical, missing).
2. Build a production-grade pipeline.
3. Tune hyperparameters with cross-validation.
4. Save the model and demonstrate inference on new data.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import os
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn: Preprocessing ──────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# ── Scikit-learn: Pipeline & Model ───────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Scikit-learn: Evaluation ─────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, roc_curve,
    precision_recall_curve
)

# ── Persistence ──────────────────────────────────────────────────
import joblib

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Revolut Churn Dataset

We simulate **30,000 Revolut users** with mixed data types and missing values.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_USERS = 30_000
CHURN_RATE = 0.18  # 18% churn rate

print(f"Generating Revolut churn dataset:")
print(f"  Users      : {N_USERS:,}")
print(f"  Churn rate : {CHURN_RATE*100:.0f}%")

# ── Features ─────────────────────────────────────────────────────
np.random.seed(42)

# Numeric features
age = np.random.randint(18, 75, N_USERS)
tenure_months = np.random.exponential(18, N_USERS).clip(0, 60)
total_transactions = np.random.poisson(50, N_USERS)
avg_transaction_amount = np.random.lognormal(3.5, 1.0, N_USERS)
num_support_tickets = np.random.poisson(1.5, N_USERS)
days_since_last_login = np.random.exponential(10, N_USERS).clip(0, 180)
credit_score = np.random.normal(650, 80, N_USERS).clip(300, 850)

# Categorical features
plan = np.random.choice(["Standard", "Premium", "Metal", "Ultra"], N_USERS, p=[0.60, 0.25, 0.10, 0.05])
country = np.random.choice(["UK", "DE", "FR", "ES", "IT", "PL", "NL"], N_USERS, p=[0.40, 0.15, 0.12, 0.10, 0.08, 0.08, 0.07])
device = np.random.choice(["iOS", "Android", "Web"], N_USERS, p=[0.45, 0.40, 0.15])

# ── Target: Churn probability (logistic function) ────────────────
log_odds = (
    -2.0
    - 0.02 * (age - 40)
    - 0.05 * tenure_months
    + 0.02 * days_since_last_login
    + 0.3 * num_support_tickets
    - 0.001 * total_transactions
    + 0.5 * (plan == "Standard")
    - 0.5 * (plan == "Metal")
    + np.random.normal(0, 1, N_USERS)
)
prob_churn = 1 / (1 + np.exp(-log_odds))
churned = (prob_churn > np.percentile(prob_churn, (1 - CHURN_RATE) * 100)).astype(int)

# ── Assemble DataFrame ───────────────────────────────────────────
df = pd.DataFrame({
    "age": age,
    "tenure_months": tenure_months,
    "total_transactions": total_transactions,
    "avg_transaction_amount": avg_transaction_amount,
    "num_support_tickets": num_support_tickets,
    "days_since_last_login": days_since_last_login,
    "credit_score": credit_score,
    "plan": plan,
    "country": country,
    "device": device,
    "churned": churned,
})

# Inject missing values (realistic data quality)
df.loc[np.random.random(N_USERS) < 0.05, "age"] = np.nan
df.loc[np.random.random(N_USERS) < 0.08, "credit_score"] = np.nan
df.loc[np.random.random(N_USERS) < 0.03, "tenure_months"] = np.nan
df.loc[np.random.random(N_USERS) < 0.04, "plan"] = np.nan

print(f"\n✓ Shape: {df.shape}")
print(f"  Churn rate: {df['churned'].mean() * 100:.1f}%")
print(f"  Missing values: {df.isnull().sum().sum()}")

df.head(10)

In [ ]:
# Data types and missing values
info_df = pd.DataFrame({
    "Dtype": df.dtypes,
    "Non-Null": df.count(),
    "Missing": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2),
})
print(info_df)

---
## §4 · Exploratory Data Analysis

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Churn distribution
fig = px.pie(
    names=["Active", "Churned"],
    values=[df["churned"].value_counts()[0], df["churned"].value_counts()[1]],
    hole=0.4,
    title="Customer Churn Distribution",
    color_discrete_sequence=["#00CC96", "#EF553B"],
)
fig.update_layout(height=380)
fig.show()

In [ ]:
# Feature distributions by churn status
fig = make_subplots(rows=2, cols=3,
                    subplot_titles=["Age", "Tenure (months)", "Total Transactions",
                                    "Avg Transaction Amount", "Support Tickets", "Days Since Last Login"])

numeric_cols = ["age", "tenure_months", "total_transactions",
                "avg_transaction_amount", "num_support_tickets", "days_since_last_login"]

for i, col in enumerate(numeric_cols, 1):
    row = (i - 1) // 3 + 1
    col_idx = (i - 1) % 3 + 1
    
    fig.add_trace(go.Box(
        x=df["churned"], y=df[col],
        name=col, marker_color=["#00CC96", "#EF553B"][i % 2],
    ), row=row, col=col_idx)

fig.update_layout(height=600, showlegend=False, title_text="Feature Distributions by Churn Status")
fig.update_xaxes(title_text="Churned (0=No, 1=Yes)")
fig.show()

In [ ]:
# Categorical feature analysis
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["Plan", "Country", "Device"],
                    specs=[[{"type": "bar"}, {"type": "bar"}, {"type": "bar"}]])

for i, col in enumerate(["plan", "country", "device"], 1):
    churn_rate_by_cat = df.groupby(col)["churned"].mean().sort_values(ascending=False)
    fig.add_trace(go.Bar(
        x=churn_rate_by_cat.index,
        y=churn_rate_by_cat.values,
        name=col,
        marker_color=px.colors.qualitative.Set2[:len(churn_rate_by_cat)],
    ), row=1, col=i)

fig.update_layout(height=400, showlegend=False, title_text="Churn Rate by Categorical Features")
fig.update_yaxes(title_text="Churn Rate", tickformat=".0%")
fig.show()

---
## §5 · Building the Pipeline

### 5a — Define Preprocessing Steps

We separate features by type and apply appropriate transformers.

In [ ]:
# Separate features and target
X = df.drop(columns=["churned"])
y = df["churned"]

# Identify feature types
numeric_features = ["age", "tenure_months", "total_transactions",
                    "avg_transaction_amount", "num_support_tickets",
                    "days_since_last_login", "credit_score"]

categorical_features = ["plan", "country", "device"]

print(f"Numeric features    : {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Total features      : {len(numeric_features) + len(categorical_features)}")

# ── Numeric transformer ──────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),      # fill missing with median
    ("scaler", StandardScaler()),                        # standardize to mean=0, std=1
])

# ── Categorical transformer ──────────────────────────────────────
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),  # fill missing
    ("onehot", OneHotEncoder(handle_unknown="ignore")),                     # one-hot encode
])

# ── Combine into preprocessor ────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",  # drop any unspecified columns
)

print("\n✓ Preprocessor defined")
print("  Numeric: Impute (median) → Scale (standard)")
print("  Categorical: Impute (constant) → One-Hot Encode")

### 5b — Full Pipeline: Preprocessing + Feature Selection + Model

In [ ]:
# ── Build the complete pipeline ──────────────────────────────────
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=mutual_info_classif, k=15)),  # select top 15 features
    ("classifier", XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False,
    )),
])

print("=" * 70)
print("PIPELINE STRUCTURE")
print("=" * 70)
for i, (name, step) in enumerate(pipeline.steps, 1):
    print(f"{i}. {name:20s}: {step.__class__.__name__}")

print("\n✓ Pipeline ready for training")

---
## §6 · Cross-Validation

Train and validate the pipeline using 5-fold stratified cross-validation.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f"Train: {len(y_train):,} samples ({y_train.sum():,} churned)")
print(f"Test : {len(y_test):,} samples ({y_test.sum():,} churned)")

# ── Cross-validation ─────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nRunning 5-fold cross-validation …")
t0 = time.time()
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
time_cv = time.time() - t0

print(f"✓ Completed in {time_cv:.1f}s")
print(f"\nCross-Validation Results:")
print(f"  Fold 1: {cv_scores[0]:.4f}")
print(f"  Fold 2: {cv_scores[1]:.4f}")
print(f"  Fold 3: {cv_scores[2]:.4f}")
print(f"  Fold 4: {cv_scores[3]:.4f}")
print(f"  Fold 5: {cv_scores[4]:.4f}")
print(f"\n  Mean AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Visualize
fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f"Fold {i}" for i in range(1, 6)],
    y=cv_scores,
    marker_color=["#636EFA", "#EF553B", "#00CC96", "#FFA15A", "#AB63FA"],
    text=cv_scores.round(4),
    textposition="outside",
))
fig.add_hline(y=cv_scores.mean(), line_dash="dash", line_color="black",
              annotation_text=f"Mean: {cv_scores.mean():.4f}")

fig.update_layout(
    title="5-Fold Cross-Validation AUC Scores",
    yaxis_title="ROC-AUC",
    height=400,
    yaxis_range=[cv_scores.min() - 0.01, cv_scores.max() + 0.02],
)
fig.show()

---
## §7 · Hyperparameter Tuning with Pipeline

Use `GridSearchCV` to tune pipeline hyperparameters (including preprocessing).

In [ ]:
# Define parameter grid (small for speed)
param_grid = {
    "feature_selection__k": [10, 15, 20],
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.05, 0.1, 0.2],
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Grid Search:")
print(f"  Combinations: {total_combinations}")
print(f"  CV folds    : 3 (reduced for speed)")
print(f"  Total fits  : {total_combinations * 3}")

# Run grid search
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=0,
)

print("\nRunning grid search …")
t0 = time.time()
grid_search.fit(X_train, y_train)
time_grid = time.time() - t0

print(f"✓ Completed in {time_grid:.1f}s")
print(f"\nBest parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param:40s}: {value}")

print(f"\nBest CV AUC: {grid_search.best_score_:.4f}")

# Use best model
best_pipeline = grid_search.best_estimator_

---
## §8 · Final Model Evaluation

Evaluate the best pipeline on the **held-out test set**.

In [ ]:
# Predict on test set
y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

# Metrics
print("=" * 70)
print("FINAL MODEL EVALUATION (Test Set)")
print("=" * 70)
print(f"\nAccuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives : {cm[0, 0]:,}")
print(f"False Positives: {cm[0, 1]:,}")
print(f"False Negatives: {cm[1, 0]:,}")
print(f"True Positives : {cm[1, 1]:,}")

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode="lines",
    name=f"ROC Curve (AUC = {auc_score:.4f})",
    line=dict(color="#636EFA", width=2.5),
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Random Classifier",
    line=dict(color="gray", width=1, dash="dash"),
))

fig.update_layout(
    title="ROC Curve",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=450,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

In [ ]:
# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc_score = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall, y=precision,
    mode="lines",
    name=f"PR Curve",
    line=dict(color="#00CC96", width=2.5),
))

fig.update_layout(
    title="Precision-Recall Curve",
    xaxis_title="Recall",
    yaxis_title="Precision",
    height=450,
)
fig.show()

---
## §9 · Model Persistence & Inference

### 9.1 — Save the Pipeline

In [ ]:
# Save the entire pipeline (preprocessing + model)
model_path = "churn_prediction_pipeline.joblib"
joblib.dump(best_pipeline, model_path)

file_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"✓ Pipeline saved to {model_path}")
print(f"  File size: {file_size_mb:.2f} MB")

print("\n💡 The saved pipeline includes:")
print("   - Preprocessing (imputer, scaler, encoder)")
print("   - Feature selection")
print("   - Trained model")
print("   → Everything needed for inference in a single file!")

### 9.2 — Load and Use for Inference

In [ ]:
# Load the pipeline
loaded_pipeline = joblib.load(model_path)
print(f"✓ Pipeline loaded from {model_path}")

# ── Simulate new data (inference) ────────────────────────────────
new_customers = pd.DataFrame({
    "age": [25, 45, 35, 60, 28],
    "tenure_months": [6, 24, 12, 36, 3],
    "total_transactions": [20, 150, 80, 200, 10],
    "avg_transaction_amount": [50.0, 200.0, 100.0, 500.0, 30.0],
    "num_support_tickets": [3, 0, 1, 0, 5],
    "days_since_last_login": [30, 2, 5, 1, 60],
    "credit_score": [600, 750, 680, 800, 550],
    "plan": ["Standard", "Metal", "Premium", "Ultra", "Standard"],
    "country": ["UK", "DE", "FR", "UK", "ES"],
    "device": ["iOS", "Android", "Web", "iOS", "Android"],
})

print("\nNew customers (inference data):")
print(new_customers.to_string(index=False))

# ── Predict ──────────────────────────────────────────────────────
churn_prob = loaded_pipeline.predict_proba(new_customers)[:, 1]
churn_pred = loaded_pipeline.predict(new_customers)

new_customers["Churn Probability"] = churn_prob
new_customers["Predicted Churn"] = churn_pred

print("\nPredictions:")
print(new_customers.to_string(index=False))

print("\n💡 The pipeline automatically:")
print("   - Imputes missing values")
print("   - Scales numeric features")
print("   - Encodes categorical features")
print("   - Selects top features")
print("   - Makes predictions")

In [ ]:
# Business action: prioritize retention campaigns
high_risk = new_customers[new_customers["Churn Probability"] > 0.5].sort_values(
    "Churn Probability", ascending=False
)

if len(high_risk) > 0:
    print("=" * 70)
    print("HIGH-RISK CUSTOMERS (Churn Probability > 50%)")
    print("=" * 70)
    print(high_risk.to_string(index=False))
    print(f"\n→ Action: Send personalized retention offers to {len(high_risk)} customers")
else:
    print("✅ No high-risk customers detected")

---
## §10 · Visualization Dashboard

In [ ]:
# ── 10.1 Feature Importance ──────────────────────────────────────
# Get feature names after preprocessing and selection
preprocessor = best_pipeline.named_steps["preprocessor"]
feature_selector = best_pipeline.named_steps["feature_selection"]
classifier = best_pipeline.named_steps["classifier"]

# Get feature names
num_features = preprocessor.named_transformers_["num"].get_feature_names_out(numeric_features)
cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
all_features = np.concatenate([num_features, cat_features])

# Get selected features
selected_mask = feature_selector.get_support()
selected_features = all_features[selected_mask]

# Get feature importance from XGBoost
importance = classifier.feature_importances_

importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": importance,
}).sort_values("Importance", ascending=True)

fig = px.bar(
    importance_df.tail(15),
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top 15 Feature Importances (After Pipeline)",
    color="Importance",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=500, yaxis=dict(autorange="reversed"))
fig.show()

print(f"✓ Selected {len(selected_features)} features from {len(all_features)} total")

In [ ]:
# ── 10.2 Pipeline Performance Summary ────────────────────────────
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob),
    ],
}).round(4)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=summary["Metric"],
    y=summary["Score"],
    marker_color=["#636EFA", "#EF553B", "#00CC96", "#FFA15A", "#AB63FA"],
    text=summary["Score"].round(4),
    textposition="outside",
))

fig.update_layout(
    title="Model Performance Summary",
    yaxis_title="Score",
    height=420,
    yaxis_range=[0, 1.1],
)
fig.show()

print("\n💡 Balanced precision and recall indicate a well-calibrated model")
print("   ROC-AUC > 0.85 is considered excellent for churn prediction")

In [ ]:
# ── 10.3 Business Impact Estimate ────────────────────────────────
# Estimate business value of the model
n_test = len(y_test)
actual_churn = y_test.sum()
predicted_churn = y_pred.sum()
true_positives = cm[1, 1]

# Assumptions
avg_customer_value = 500  # £ per year
retention_campaign_cost = 50  # £ per customer
retention_success_rate = 0.30  # 30% of targeted customers stay

print("=" * 70)
print("BUSINESS IMPACT ESTIMATE")
print("=" * 70)
print(f"\nTest set statistics:")
print(f"  Total customers     : {n_test:,}")
print(f"  Actual churn        : {actual_churn:,} ({actual_churn/n_test*100:.1f}%)")
print(f"  Predicted churn     : {predicted_churn:,}")
print(f"  True positives      : {true_positives:,} (correctly identified churners)")

print(f"\nAssumptions:")
print(f"  Avg customer value  : £{avg_customer_value:,}/year")
print(f"  Campaign cost       : £{retention_campaign_cost}/customer")
print(f"  Retention success   : {retention_success_rate*100:.0f}%")

# Calculate savings
customers_saved = true_positives * retention_success_rate
revenue_saved = customers_saved * avg_customer_value
campaign_cost = predicted_churn * retention_campaign_cost
net_value = revenue_saved - campaign_cost

print(f"\nEstimated annual impact (extrapolated to full customer base):")
print(f"  Customers potentially saved: {int(customers_saved):,}")
print(f"  Revenue preserved         : £{int(revenue_saved):,}")
print(f"  Campaign cost             : £{int(campaign_cost):,}")
print(f"  Net value                 : £{int(net_value):,}")

print(f"\n💡 Even a modest churn prediction model can save millions in revenue")
print("   by enabling targeted, cost-effective retention campaigns")

---
## §11 · Key Business Takeaway

> ### 🎯 Action Items for a Data Scientist
>
> | Pipeline Component | Purpose | Key Class |
> |--------------------|---------|-----------|
> | **Preprocessor** | Handle missing values, scale, encode | `ColumnTransformer` |
> | **Feature Selection** | Reduce dimensionality, improve speed | `SelectKBest`, `SelectFromModel` |
> | **Classifier** | Predict target | `XGBClassifier`, `LogisticRegression` |
> | **Pipeline** | Chain all steps together | `Pipeline` |
> | **Persistence** | Save/load for deployment | `joblib.dump/load` |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Always use pipelines** — never preprocess data outside the pipeline:
>    ```python
>    # WRONG: data leakage
>    scaler = StandardScaler()
>    X_scaled = scaler.fit_transform(X)  # fits on all data
>    X_train, X_test = train_test_split(X_scaled)
>    
>    # RIGHT: pipeline prevents leakage
>    pipeline = Pipeline([("scaler", StandardScaler()), ("model", XGBClassifier())])
>    pipeline.fit(X_train, y_train)  # fits scaler on train only
>    ```
>
> 2. **Save the entire pipeline**, not just the model:
>    ```python
>    joblib.dump(pipeline, "model.joblib")  # includes preprocessing
>    ```
>
> 3. **Version your models** with metadata:
>    ```python
>    metadata = {
>        "model": pipeline,
>        "version": "1.2.0",
>        "trained_date": "2025-06-15",
>        "metrics": {"auc": 0.87},
>    }
>    joblib.dump(metadata, "model_v1.2.0.joblib")
>    ```
>
> 4. **Monitor model drift** in production:
>    - Track prediction distribution over time.
>    - Retrain when performance degrades (see Module 24: Data Drift).
>
> ### 📌 Production Checklist
>
> ```
> ✓ Pipeline includes all preprocessing steps
> ✓ Cross-validation used (no data leakage)
> ✓ Hyperparameters tuned with GridSearchCV
> ✓ Model saved with joblib (includes preprocessing)
> ✓ Inference tested on new data
> ✓ Feature importance documented
> ✓ Business impact estimated
> ✓ Model version tracked
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **Pipelines prevent data leakage** — the #1 mistake in ML projects.
> 2. **Save the entire pipeline** — preprocessing + model in one file.
> 3. **Cross-validation** is mandatory — never evaluate on training data.
> 4. **Business impact** matters more than AUC — estimate revenue saved.
> 5. **Version and monitor** — models degrade over time (data drift).

In [ ]:
# Cleanup
if os.path.exists(model_path):
    os.remove(model_path)
    print(f"✓ Removed {model_path}")

print("\n✅ Module 10 complete!")
print("   Track 1 (Data Science Foundations) is now complete!")
print("   Next: Track 2 — Revolut Fintech Specialization (Modules 11–24)")